# Runaway Agent Loop — Velocity Kill + Cascading Void

[![PyPI](https://img.shields.io/pypi/v/figuard.svg)](https://pypi.org/project/figuard/)
[![GitHub](https://img.shields.io/badge/github-figuard%2Ffiguard--core-blue)](https://github.com/figuard/figuard-core)

## The problem this solves

A ReAct-style research agent is told: *"find everything about AI cost overruns."*

It loops. Search → summarize → decide more info needed → search again.
In production with no guard, this runs 50–200 iterations before you notice.
By then the API bill has already happened.

## What this notebook demonstrates

**FiGuard velocity controls** catch the runaway pattern before it compounds:
- `velocity_max_per_minute=5` — at most 5 authorization events per 60-second window
- The 6th call in under a minute is denied: `VELOCITY_LIMIT_EXCEEDED`

**Cascading void** cleans up atomically when the loop is killed:
- `void_tree(root_event_id)` voids the orchestrator + all child agents in one server-side transaction
- No dangling reservations, no partial state — everything released at once

**OTEL traces in Langfuse** make the whole sequence visible:
- The velocity denial appears as a `GUARDRAIL` ERROR span
- The void cascade appears as a `GUARDRAIL` span under the orchestrator
- Three successful searches + the summarizer reservation show as `AGENT` spans

**Scenario agents:**
- `orchestrator` — creates the pipeline, monitors for denials, calls void_tree on failure
- `searcher` — fires web search requests (the runaway agent)
- `summarizer` — had a pending reservation for summarizing batch 1 (gets voided in the cascade)

**No LLM calls, no paid API keys** — FiGuard sandbox connects automatically.

## Step 1 — Install dependencies

In [ ]:
%pip install -qU figuard[opentelemetry] langfuse

## Step 2 — Configure Langfuse credentials

Create a project at [cloud.langfuse.com](https://cloud.langfuse.com), then go to
**Project Settings → API Keys → Create new key**.

> US region: `https://us.cloud.langfuse.com`  
> EU region: `https://eu.cloud.langfuse.com`

In [ ]:
import os
import getpass

if not os.environ.get("LANGFUSE_PUBLIC_KEY"):
    os.environ["LANGFUSE_PUBLIC_KEY"] = getpass.getpass("Langfuse public key (pk-lf-...): ")
if not os.environ.get("LANGFUSE_SECRET_KEY"):
    os.environ["LANGFUSE_SECRET_KEY"] = getpass.getpass("Langfuse secret key (sk-lf-...): ")

os.environ.setdefault("LANGFUSE_HOST", "https://us.cloud.langfuse.com")
os.environ["FIGUARD_SUPPRESS_SANDBOX_WARNING"] = "1"

print("Credentials set.")

## Step 3 — Connect Langfuse

Langfuse v4 uses OTEL internally. FiGuard picks up the active OTEL provider automatically —
every `authorize()`, `confirm_event()`, `fail_event()`, and `void_tree()` call emits a span.

In [ ]:
from langfuse import Langfuse

lf = Langfuse(
    public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
    secret_key=os.environ["LANGFUSE_SECRET_KEY"],
    host=os.environ["LANGFUSE_HOST"],
)

assert lf.auth_check(), "Langfuse auth failed — check your keys and host"
print(f"Connected to Langfuse at {os.environ['LANGFUSE_HOST']}")
print("Velocity denials will appear as GUARDRAIL ERROR spans.")
print("void_tree cascade will appear as a GUARDRAIL span under the orchestrator.")

## Step 4 — Create the research pipeline budget

`velocity_max_per_minute=4` is the circuit breaker.

A normal research iteration uses at most 3–4 calls per minute (a search or two, a summarization).
Setting the limit to 4 means:
- Normal operation: fine — 3 searches + summarization fits exactly
- A loop firing a 5th call in under 60 seconds: **blocked**

In production you'd tune this to 2–3× your agent's normal call rate.
The point is that a runaway loop always exceeds normal rate by an order of magnitude.

In [ ]:
from figuard import FiGuardClient

client = FiGuardClient()  # zero-config: public sandbox

budget = client.create_budget(
    user_id="research_pipeline",
    total_limit=50.00,
    currency="USD",
    expires_in="1h",
    velocity_max_per_minute=4,   # circuit breaker: 3 searches + 1 summarization = normal; 5th = denied
)
session_token = budget.primary_token.session_token

print(f"Budget:                {budget.id}")
print(f"Total limit:           ${budget.total_limit:,.2f}")
print(f"velocity_max_per_min:  {budget.velocity_max_per_minute}")
print()
print("Normal iteration: search × 3 + summarize × 1 = 4 events — fits exactly.")
print("Runaway loop: tries a 5th search in the same minute — denied.")

## Step 5 — Run the pipeline (loop goes rogue at iteration 4)

| Event | Agent | Decision | Velocity | Note |
|---|---|---|---|---|
| web_search #1 | searcher | AUTHORIZED → CONFIRMED | 1/4 | $0.05 spent |
| web_search #2 | searcher | AUTHORIZED → CONFIRMED | 2/4 | $0.05 spent |
| web_search #3 | searcher | AUTHORIZED → CONFIRMED | 3/4 | $0.05 spent |
| summarize batch-1 | summarizer | AUTHORIZED | 4/4 — at limit | $0.20 pending |
| web_search #4 | searcher | **DENIED** | 5/4 — **VELOCITY_LIMIT_EXCEEDED** | loop caught |
| void summarizer | orchestrator | VOIDED | — | $0.20 released |

**Confirmed searches are not voided** — that work already happened, $0.15 is spent.
Only the summarizer's pending reservation ($0.20) is released, because that work
never started.

In [ ]:
print("Starting research pipeline...\n")

with lf.start_as_current_observation(
    as_type="agent",
    name="research-pipeline-orchestrator",
    input={"task": "Research AI cost overruns", "budget_limit": 50.00},
) as root_obs:

    # ── Searcher: 3 searches — authorized then immediately confirmed ─────────
    with lf.start_as_current_observation(
        as_type="agent", name="searcher",
        metadata={"velocity_limit": "4/min"},
    ) as searcher_obs:
        for i, query in enumerate(search_queries, 1):
            r = client.authorize(
                session_token=session_token,
                agent_id="searcher",
                action_type="web_search",
                description=f"Search #{i}: {query}",
                requested_quantity=0.05,
                currency="USD",
            )
            client.confirm_event(r.event_id, confirmed_quantity=0.05)
            print(f"[searcher]   web_search #{i}    CONFIRMED  $0.05 spent  (velocity {i}/4)")
        searcher_obs.update(output={"searches_completed": 3, "total_confirmed": 0.15})

    # ── Summarizer: pending reservation — work hasn't started ────────────────
    with lf.start_as_current_observation(
        as_type="agent", name="summarizer",
    ) as summ_obs:
        summ_auth = client.authorize(
            session_token=session_token,
            agent_id="summarizer",
            action_type="summarize",
            description="Summarize search batch 1 (3 results)",
            requested_quantity=0.20,
            currency="USD",
        )
        print(f"[summarizer] summarize       AUTHORIZED $0.20 reserved  (velocity 4/4 — AT LIMIT)")
        summ_obs.update(output={"status": "pending — waiting for more search results"})

    print()
    print("  4/4 velocity slots used. Next call this minute: DENIED.")
    print()

    # ── Searcher tries a 4th search — velocity limit exceeded ────────────────
    # Clear the ambient event ID so the denied call is not chained as a child
    # of the summarizer. It stands alone as a sibling (same pipeline, separate action).
    with lf.start_as_current_observation(
        as_type="agent", name="searcher-loop-iteration-4",
    ) as loop_obs:
        denial = client.authorize(
            session_token=session_token,
            agent_id="searcher",
            action_type="web_search",
            description="Search #4: loop iteration 4 — agent decided more info needed",
            requested_quantity=0.05,
            currency="USD",
            parent_event_id=None,   # explicit None: break ContextVar chain, denied call is a root
        )
        print(f"[searcher]   web_search #4   {denial.decision}   reason={denial.denial_reason}  (velocity 5/4)")

        with lf.start_as_current_observation(
            as_type="guardrail",
            name="figuard.velocity_kill",
            level="ERROR",
            metadata={
                "denial_reason": denial.denial_reason,
                "agent": "searcher",
                "velocity_limit": "4/min",
                "calls_attempted": 5,
            },
            output={"decision": "DENIED", "reason": denial.denial_reason},
        ):
            pass
        loop_obs.update(output={"status": "killed by velocity guard"})

    # ── Orchestrator: void only the pending summarizer reservation ───────────
    print()
    print("[orchestrator] voiding pending summarizer reservation...")

    with lf.start_as_current_observation(
        as_type="guardrail",
        name="figuard.void_pending",
        metadata={"reason": "VELOCITY_LIMIT_EXCEEDED"},
    ) as void_obs:
        client.void_event(summ_auth.event_id, reason="VELOCITY_LIMIT_EXCEEDED")
        void_obs.update(output={
            "voided": "summarizer",
            "quantity_released": 0.20,
            "confirmed_searches_untouched": 3,
        })

    print(f"[void]       summarizer voided      $0.20 released")
    print(f"[preserved]  3 confirmed searches   $0.15 — already spent, not touched")

    root_obs.update(output={
        "status": "cancelled",
        "reason": "VELOCITY_LIMIT_EXCEEDED",
        "confirmed_spend": 0.15,
        "pending_released": 0.20,
    })

lf.flush()
print()
print("Done. Open Langfuse → Tracing and look for: research-pipeline-orchestrator")
print("In the FiGuard spend tree: 3 CONFIRMED + 1 VOIDED (not 5 VOIDED)")

## Step 6 — What you see in Langfuse

Open **Langfuse → Tracing → research-pipeline-orchestrator**.

```
research-pipeline-orchestrator  [AGENT]
  ├── searcher                  [AGENT]    — 3 searches, confirmed
  ├── summarizer                [AGENT]    — pending, voided by cascade
  ├── searcher-loop-iteration-4 [AGENT]
  │     └── figuard.velocity_kill  [GUARDRAIL ERROR]  ← the circuit breaker firing
  └── figuard.void_tree         [GUARDRAIL]           ← atomic cascade kill
        voided_count: 5
        quantity_released: $1.35
```

The graph view shows the exact moment the loop was killed:
- Three green agent nodes (normal execution)
- One red GUARDRAIL node (velocity denial)
- One final GUARDRAIL node (void_tree) before `__end__`

**What this tells you that logs alone can't:**
- Which specific agent tripped the velocity limit (searcher, not orchestrator)
- That the summarizer had a pending reservation that was released (not just abandoned)
- The exact dollar amount returned to the budget: $1.35
- The causal chain: loop → denial → cascade, all in one trace

## Step 7 — Inspect the FiGuard ledger

The ledger is the append-only audit trail. Every authorization, confirmation, and void
is recorded — 100% of events, not sampled.

In [ ]:
page = client.get_ledger(budget.id, page=0, size=20)

print(f"Budget: {budget.id}")
print(f"Total ledger events: {page.total_elements}")
print()
print(f"  {'DECISION':<18}  {'AMOUNT':>8}  {'AGENT':<14}  DETAIL")
print(f"  {'-'*18}  {'-'*8}  {'-'*14}  {'-'*35}")

for ev in page.events:
    if ev.decision in ("AUTHORIZED", "CONFIRMED"):
        icon = "  [OK] "
    elif ev.decision in ("VOIDED",):
        icon = "  [---]"
    else:
        icon = "  [X]  "
    amount = ev.confirmed_quantity or ev.requested_quantity or 0
    detail = ev.denial_reason or ev.action_type or ""
    print(f"{icon}{ev.decision:<14}  ${amount:>7.2f}  {ev.agent_id:<14}  {detail}")

## Step 8 — Final budget state

In [ ]:
final = client.get_budget(budget.id)

print("Research pipeline budget — final state")
print(f"  Total limit:   ${final.total_limit:,.2f}")
print(f"  Confirmed:     ${final.quantity_spent:,.2f}    ← only the 3 confirmed searches")
print(f"  Reserved:      ${final.quantity_reserved:,.2f}    ← cleared by void_tree")
print(f"  Available:     ${final.available_quantity:,.2f}")
print()
print("The 3 confirmed searches cost $0.15 total.")
print("The orchestrator reservation, summarizer reservation, and 3 search")
print("reservations were all released — $1.35 returned to the budget.")
print()
print("Without FiGuard: the loop would have continued at $0.05/search.")
print("At 200 iterations (typical runaway): $10.00 in search costs alone,")
print("plus the LLM tokens on each summarization step.")

## What this demonstrates

| Capability | Where it shows up in this notebook |
|---|---|
| **Velocity controls** | `velocity_max_per_minute=5` blocks search #4 — the 6th event in the window |
| **Cascading void** | `void_tree()` atomically releases 5 pending reservations in one call |
| **Causal chain** | All sub-agent events linked via `parent_event_id` — void_tree walks the tree server-side |
| **OTEL in Langfuse** | Denial + cascade both visible as `GUARDRAIL` spans in the execution graph |
| **Append-only ledger** | Every event recorded: authorized, confirmed, denied, voided |
| **Zero-config sandbox** | Runs with no API key — public sandbox connects automatically |

### Tuning velocity controls for your agent

```python
budget = client.create_budget(
    ...
    velocity_max_per_minute=5,        # max events per 60-second rolling window
    velocity_max_amount_per_hour=10.0, # max dollars per hour
    velocity_max_per_day=100,          # max events per calendar day
)
```

Set the limit to 2–3× your agent's normal call rate. A runaway loop always
exceeds that by an order of magnitude. Normal operation never notices the cap.

### Self-hosting

```bash
git clone https://github.com/figuard/figuard-core
cd figuard-core
docker compose up -d
```

Then set `FIGUARD_BASE_URL=http://localhost:8080` and `FIGUARD_API_KEY=<your-key>`
before importing `FiGuardClient`.